In [9]:
import os
import sys
import time
import pandas as pd
import numpy  as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl
import warnings
warnings.filterwarnings('ignore')

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import precision_recall_curve, classification_report
from sklearn.datasets import make_classification
from xgboost import XGBClassifier
from xgboost import plot_importance
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression

import lightgbm as lgbm

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

import HyperParams as HP

from utils import user_utils as uu
from utils import preprocessing as pp
from utils import data_sampling as ds
from utils import model_utils as mu
from utils import modeling as mo

In [ ]:
# 결과받을 딕셔너리
results = {}

In [10]:
#1. 데이터 로딩
raw_df = pp.ccf_load_data()


데이터 로드 성공: (284807, 31)


In [ ]:
#2. Time 컬럼 삭제 , 데이터,타겟 분리
X_features, y_target = pp.split_features_target(raw_df, cols= 'Time')



In [ ]:
#3. 이상치를 경계값으로 치환
cap_X_feature = pp.cap_outliers(X_features)

In [ ]:
#4. 학습/테스트 데이터 분리
X_train, X_test, y_train, y_test = pp.data_split(X_features, y_target)


In [ ]:
#4.1. 만약에 스케일을 할꺼면 여기서 스케일적용
X_scaled, y_scaled, scaler = pp.scale_data(X_train, X_test)

In [ ]:
#5.1 학습/검증 데이터 분리
X_tr, X_val, y_tr, y_val = pp.data_split(X_train, y_train, size=0.4)

In [ ]:
#5.2 스케일 적용한 경우 학습/검증 데이터 분리
X_tr, X_val, y_tr, y_val = pp.data_split(X_scaled, y_scaled, size=0.4)

In [ ]:
#6 하이퍼파라미터
tuner = uu.HyperOptTuner(max_evals=100, random_state=23)

# catboost 스페이스 생성
catboost_search_space = {
        'iterations': hp.quniform('iterations', 100, 1000, 50),
        'depth': hp.quniform('depth', 3, 10, 1),
        'learning_rate': hp.uniform('learning_rate', 0.01, 0.03),
        'l2_leaf_reg': hp.quniform('l2_leaf_reg', 2, 30, 1),
        'border_count': hp.quniform('border_count', 32, 255, 1)
    }
# 모델 생성
catboost = CatBoostClassifier()
# 파라미터
best_params, best_catboost, trials, exec_time = tuner.tune(
    catboost, X_tr, y_tr, X_val, y_val, catboost_search_space
    )
# best모델로 결과출력
# 모델명 규칙 : 2~3자리 모델명 + _ho_best
model_name = 'cb_ho_best'
results[model_name] =uu.get_model_train_eval(
    best_catboost, model_name, X_train, X_test, y_train, y_test, best_params
    )


In [ ]:
#7 시각화
mo.model_metrics_graph(results, 'cb모델 성능지표 비교')